# Step 1: Data Download & Ingestion
## AI-Assisted Health Risk Prediction for Life Insurance Underwriting

This notebook downloads:
1. **NHANES survey data** (Demographics, Examination, Laboratory, Questionnaire) for cycles **2007-2008, 2009-2010, and 2011-2012**
2. **CDC Public-Use Linked Mortality Files** (mortality follow-up through Dec 31, 2019)

**Why these cycles?** Older survey cycles linked to the 2019 mortality cutoff give **7-12 years of follow-up**, which yields far more observed deaths (stronger training signal) than recent cycles, while staying current enough to keep measurements reasonably comparable.

Data is merged using SEQN (respondent sequence number) and saved locally.

## 1.1 Setup & Configuration

In [1]:
# Install required packages (uncomment if needed)
# !pip install pandas pyreadstat requests

import pandas as pd
import numpy as np
import requests
import os
import io
from pathlib import Path

print(f'pandas version: {pd.__version__}')
print(f'numpy version: {np.__version__}')

pandas version: 3.0.3
numpy version: 2.4.6


In [2]:
# Configuration
PROJECT_DIR = Path('..')  # Current directory
DATA_DIR = PROJECT_DIR / 'data'
RAW_DIR = DATA_DIR / 'raw'
PROCESSED_DIR = DATA_DIR / 'processed'

# Create directories
for d in [DATA_DIR, RAW_DIR, PROCESSED_DIR]:
    d.mkdir(parents=True, exist_ok=True)
    print(f'Directory ready: {d}')

Directory ready: ..\data
Directory ready: ..\data\raw
Directory ready: ..\data\processed


## 1.2 Define NHANES Data Sources

We download XPT (SAS Transport) files for key health variables relevant to
mortality/insurance underwriting from three NHANES cycles: **2007-2008**, **2009-2010**, and **2011-2012**.

Note: NHANES file names carry a cycle suffix (`_E`=2007-08, `_F`=2009-10, `_G`=2011-12).

In [3]:
# NHANES XPT file URLs organized by component and cycle
# Selected files relevant to life insurance underwriting / mortality prediction

# Suffixes: _E = 2007-2008, _F = 2009-2010, _G = 2011-2012

NHANES_FILES = {
    # Demographics
    'demographics': {'2007-2008': 'DEMO_E', '2009-2010': 'DEMO_F', '2011-2012': 'DEMO_G'},
    # Body Measures (BMI, waist circumference)
    'body_measures': {'2007-2008': 'BMX_E', '2009-2010': 'BMX_F', '2011-2012': 'BMX_G'},
    # Blood Pressure
    'blood_pressure': {'2007-2008': 'BPX_E', '2009-2010': 'BPX_F', '2011-2012': 'BPX_G'},
    # Total Cholesterol
    'cholesterol_total': {'2007-2008': 'TCHOL_E', '2009-2010': 'TCHOL_F', '2011-2012': 'TCHOL_G'},
    # HDL Cholesterol
    'cholesterol_hdl': {'2007-2008': 'HDL_E', '2009-2010': 'HDL_F', '2011-2012': 'HDL_G'},
    # Glycohemoglobin (HbA1c)
    'glycohemoglobin': {'2007-2008': 'GHB_E', '2009-2010': 'GHB_F', '2011-2012': 'GHB_G'},
    # Complete Blood Count
    'blood_count': {'2007-2008': 'CBC_E', '2009-2010': 'CBC_F', '2011-2012': 'CBC_G'},
    # Diabetes Questionnaire
    'diabetes': {'2007-2008': 'DIQ_E', '2009-2010': 'DIQ_F', '2011-2012': 'DIQ_G'},
    # Smoking - Cigarette Use
    'smoking': {'2007-2008': 'SMQ_E', '2009-2010': 'SMQ_F', '2011-2012': 'SMQ_G'},
    # Alcohol Use
    'alcohol': {'2007-2008': 'ALQ_E', '2009-2010': 'ALQ_F', '2011-2012': 'ALQ_G'},
    # Medical Conditions
    'medical_conditions': {'2007-2008': 'MCQ_E', '2009-2010': 'MCQ_F', '2011-2012': 'MCQ_G'},
    # Physical Activity
    'physical_activity': {'2007-2008': 'PAQ_E', '2009-2010': 'PAQ_F', '2011-2012': 'PAQ_G'},
    # Kidney - Albumin & Creatinine
    'kidney_albumin_creatinine': {'2007-2008': 'ALB_CR_E', '2009-2010': 'ALB_CR_F', '2011-2012': 'ALB_CR_G'},
    # Standard Biochemistry Profile (liver, kidney markers)
    'biochemistry': {'2007-2008': 'BIOPRO_E', '2009-2010': 'BIOPRO_F', '2011-2012': 'BIOPRO_G'},
    # Health Insurance
    'health_insurance': {'2007-2008': 'HIQ_E', '2009-2010': 'HIQ_F', '2011-2012': 'HIQ_G'},
}

CYCLES = ['2007-2008', '2009-2010', '2011-2012']
print(f'Total NHANES components to download: {len(NHANES_FILES)}')
print(f'Cycles: {", ".join(CYCLES)}')
print(f'Total files: {len(NHANES_FILES) * len(CYCLES)}')

Total NHANES components to download: 15
Cycles: 2007-2008, 2009-2010, 2011-2012
Total files: 45


## 1.3 Download NHANES XPT Files

Each XPT file is a SAS Transport format file. Pandas can read these directly.
Files are cached locally to avoid re-downloading on subsequent runs.

In [4]:
def download_xpt(file_code, cycle, save_dir):
    """
    Download an NHANES XPT file and return as DataFrame.
    Files are cached locally to avoid re-downloading.
    """
    # URL format for NHANES data files uses the start year of the cycle
    start_year = cycle.split('-')[0]
    url = f'https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/{start_year}/DataFiles/{file_code}.XPT'
    local_path = save_dir / f'{file_code}.XPT'
    
    if local_path.exists():
        # Validate cached file is actually XPT (starts with 'HEADER RECORD')
        with open(local_path, 'rb') as f:
            header = f.read(20)
        if b'HEADER RECORD' not in header:
            print(f'  [INVALID CACHE - re-downloading] {file_code} ({cycle})')
            local_path.unlink()
        else:
            print(f'  [CACHED] {file_code} ({cycle})')
            df = pd.read_sas(local_path, format="xport", encoding="utf-8")
            return df
    
    print(f'  [DOWNLOADING] {file_code} ({cycle}) from {url}')
    try:
        response = requests.get(url, timeout=60)
        response.raise_for_status()
        
        # Validate response is actual XPT data, not HTML error page
        if b'HEADER RECORD' not in response.content[:80]:
            print(f'    [ERROR] Server returned non-XPT content for {file_code}')
            print(f'    URL tried: {url}')
            return None
        
        # Save locally
        with open(local_path, 'wb') as f:
            f.write(response.content)
        
        # Read into DataFrame
        df = pd.read_sas(local_path, format="xport", encoding="utf-8")
        print(f'    -> {len(df)} rows, {len(df.columns)} columns')
        return df
    except requests.exceptions.RequestException as e:
        print(f'    [ERROR] Failed to download {file_code}: {e}')
        return None

In [11]:
# Download all NHANES files
nhanes_data = {}  # {component_name: DataFrame (both cycles merged)}

for component_name, cycles in NHANES_FILES.items():
    print(f'\n--- {component_name.upper()} ---')
    cycle_dfs = []
    
    for cycle, file_code in cycles.items():
        # Create cycle-specific subdirectory
        cycle_dir = RAW_DIR / cycle
        cycle_dir.mkdir(parents=True, exist_ok=True)
        
        df = download_xpt(file_code, cycle, cycle_dir)
        if df is not None:
            df['CYCLE'] = cycle  # Tag with source cycle
            cycle_dfs.append(df)
    
    # Merge cycles
    if cycle_dfs:
        merged = pd.concat(cycle_dfs, ignore_index=True)
        nhanes_data[component_name] = merged
        print(f'  Combined: {len(merged)} rows, {len(merged.columns)} columns')

print(f'\n{"="*60}')
print(f'Successfully downloaded {len(nhanes_data)} NHANES components')


--- DEMOGRAPHICS ---
  [CACHED] DEMO_E (2007-2008)
  [CACHED] DEMO_F (2009-2010)
  [CACHED] DEMO_G (2011-2012)
  Combined: 30442 rows, 55 columns

--- BODY_MEASURES ---
  [CACHED] BMX_E (2007-2008)
  [CACHED] BMX_F (2009-2010)
  [CACHED] BMX_G (2011-2012)
  Combined: 29353 rows, 31 columns

--- BLOOD_PRESSURE ---
  [CACHED] BPX_E (2007-2008)
  [CACHED] BPX_F (2009-2010)
  [CACHED] BPX_G (2011-2012)
  Combined: 29353 rows, 28 columns

--- CHOLESTEROL_TOTAL ---
  [CACHED] TCHOL_E (2007-2008)
  [CACHED] TCHOL_F (2009-2010)
  [CACHED] TCHOL_G (2011-2012)
  Combined: 24544 rows, 4 columns

--- CHOLESTEROL_HDL ---
  [CACHED] HDL_E (2007-2008)
  [CACHED] HDL_F (2009-2010)
  [CACHED] HDL_G (2011-2012)
  Combined: 24544 rows, 4 columns

--- GLYCOHEMOGLOBIN ---
  [CACHED] GHB_E (2007-2008)
  [CACHED] GHB_F (2009-2010)
  [CACHED] GHB_G (2011-2012)
  Combined: 20835 rows, 3 columns

--- BLOOD_COUNT ---
  [CACHED] CBC_E (2007-2008)
  [CACHED] CBC_F (2009-2010)
  [CACHED] CBC_G (2011-2012)
  Combin

## 1.4 Download CDC Linked Mortality Files

The public-use Linked Mortality Files (LMF) are fixed-width ASCII `.dat` files.
Mortality follow-up through December 31, 2019.

**File layout (from CDC documentation):**
| Column | Variable | Description |
|--------|----------|-------------|
| 1-6 | SEQN | Respondent sequence number |
| 15 | ELIGSTAT | Eligibility status (1=Eligible, 2=Under 18, 3=Ineligible) |
| 16 | MORTSTAT | Mortality status (0=Alive, 1=Deceased) |
| 17-19 | UCOD_LEADING | Underlying cause of death recode |
| 20 | DIABETES | Diabetes death flag |
| 21 | HYPERTEN | Hypertension death flag |
| 22-25 | PERMTH_INT | Person-months follow-up (interview) |
| 26-29 | PERMTH_EXM | Person-months follow-up (exam) |

In [5]:
# Linked Mortality File URLs
MORT_BASE = 'https://ftp.cdc.gov/pub/Health_Statistics/NCHS/datalinkage/linked_mortality/'
MORTALITY_URLS = {
    '2007-2008': MORT_BASE + 'NHANES_2007_2008_MORT_2019_PUBLIC.dat',
    '2009-2010': MORT_BASE + 'NHANES_2009_2010_MORT_2019_PUBLIC.dat',
    '2011-2012': MORT_BASE + 'NHANES_2011_2012_MORT_2019_PUBLIC.dat',
}

# Fixed-width format specification (0-indexed, verified against actual file)
# Public-use LMF lines are 48 chars; person-months are 3-digit fields at the end.
MORTALITY_COLSPECS = [
    (0, 6),    # SEQN (public-use sequence number)
    (14, 15),  # ELIGSTAT - eligibility status
    (15, 16),  # MORTSTAT - final mortality status (0=alive, 1=deceased)
    (16, 19),  # UCOD_LEADING - underlying cause of death recode
    (19, 20),  # DIABETES - diabetes flag on death certificate
    (20, 21),  # HYPERTEN - hypertension flag on death certificate
    (42, 45),  # PERMTH_INT - person-months follow-up from interview
    (45, 48),  # PERMTH_EXM - person-months follow-up from exam
]

MORTALITY_COLUMNS = [
    'SEQN', 'ELIGSTAT', 'MORTSTAT', 'UCOD_LEADING',
    'DIABETES', 'HYPERTEN', 'PERMTH_INT', 'PERMTH_EXM'
]

In [8]:
def download_mortality(url, cycle, save_dir):
    """Download and parse a CDC Linked Mortality File (fixed-width format)."""
    filename = url.split('/')[-1]
    local_path = save_dir / filename
    
    if local_path.exists():
        print(f'  [CACHED] {filename}')
        content = local_path.read_text()
    else:
        print(f'  [DOWNLOADING] {filename}')
        try:
            # First attempt: normal SSL verification
            response = requests.get(url, timeout=60)
            response.raise_for_status()
            content = response.text
            local_path.write_text(content)
            print(f'    -> Saved to {local_path}')
        except requests.exceptions.SSLError:
            # Corporate proxy / missing intermediate CA cert on ftp.cdc.gov.
            # This is public-domain government data (no credentials sent), so we
            # retry without cert verification for this specific public download.
            print(f'    [SSL verify failed - retrying without verification (public data)]')
            try:
                import urllib3
                urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
                response = requests.get(url, timeout=60, verify=False)
                response.raise_for_status()
                content = response.text
                local_path.write_text(content)
                print(f'    -> Saved to {local_path}')
            except requests.exceptions.RequestException as e:
                print(f'    [ERROR] Failed: {e}')
                return None
        except requests.exceptions.RequestException as e:
            print(f'    [ERROR] Failed: {e}')
            return None
    
    # Parse fixed-width file
    df = pd.read_fwf(
        io.StringIO(content),
        colspecs=MORTALITY_COLSPECS,
        names=MORTALITY_COLUMNS,
        na_values=['.', ''],
    )
    
    df['CYCLE'] = cycle
    print(f'    -> {len(df)} rows parsed')
    return df

In [9]:
# Download mortality files
print('Downloading CDC Linked Mortality Files...')
print('=' * 60)

mortality_dfs = []
for cycle, url in MORTALITY_URLS.items():
    print(f'\n--- Cycle: {cycle} ---')
    cycle_dir = RAW_DIR / cycle
    cycle_dir.mkdir(parents=True, exist_ok=True)
    
    df = download_mortality(url, cycle, cycle_dir)
    if df is not None:
        mortality_dfs.append(df)

# Combine mortality data
mortality_df = pd.concat(mortality_dfs, ignore_index=True)
print(f'\n{"="*60}')
print(f'Combined mortality data: {len(mortality_df)} rows')
print(f'\nMortality status distribution:')
print(mortality_df['MORTSTAT'].value_counts(dropna=False))
print(f'\nEligibility status distribution:')
print(mortality_df['ELIGSTAT'].value_counts(dropna=False))


--- Cycle: 2007-2008 ---
  [CACHED] NHANES_2007_2008_MORT_2019_PUBLIC.dat
    -> 10149 rows parsed

--- Cycle: 2009-2010 ---
  [CACHED] NHANES_2009_2010_MORT_2019_PUBLIC.dat
    -> 10537 rows parsed

--- Cycle: 2011-2012 ---
  [CACHED] NHANES_2011_2012_MORT_2019_PUBLIC.dat
    -> 9756 rows parsed

Combined mortality data: 30442 rows

Mortality status distribution:
MORTSTAT
0.0    15963
NaN    11864
1.0     2615
Name: count, dtype: int64

Eligibility status distribution:
ELIGSTAT
1    18578
2    11823
3       41
Name: count, dtype: int64


## 1.5 Merge NHANES Data with Mortality Outcomes

Join all NHANES components together on SEQN, then link with mortality data.

In [12]:
# Start with demographics as the base (contains all respondents)
# NOTE: For the 2007-2012 cycles we use variables present in all three:
#   RIDRETH1 (race/ethnicity, all cycles) -- RIDRETH3 only exists from 2011 onward
#   DMDMARTL (marital status, older recode) -- DMDMARTZ is a newer recode
demo_cols = ['SEQN', 'CYCLE', 'RIDAGEYR', 'RIAGENDR',
             'RIDRETH1', 'DMDEDUC2', 'DMDMARTL', 'INDFMPIR']
# Only keep columns that actually exist (defensive)
demo_cols = [c for c in demo_cols if c in nhanes_data['demographics'].columns]
base_df = nhanes_data['demographics'][demo_cols].copy()
base_df.rename(columns={
    'RIDAGEYR': 'AGE',
    'RIAGENDR': 'GENDER',
    'RIDRETH1': 'RACE_ETHNICITY',
    'DMDEDUC2': 'EDUCATION',
    'DMDMARTL': 'MARITAL_STATUS',
    'INDFMPIR': 'POVERTY_INCOME_RATIO',
}, inplace=True)

print(f'Base demographics: {len(base_df)} respondents')
print(f'Age range: {base_df["AGE"].min()} - {base_df["AGE"].max()}')

Base demographics: 30442 respondents
Age range: 5.397605346934028e-79 - 80.0


In [13]:
# Merge other NHANES components on SEQN
# We select key variables from each component

merge_configs = {
    'body_measures': ['SEQN', 'BMXBMI', 'BMXWAIST', 'BMXHT', 'BMXWT'],
    'blood_pressure': ['SEQN', 'BPXSY1', 'BPXDI1', 'BPXSY2', 'BPXDI2'],
    'cholesterol_total': ['SEQN', 'LBXTC'],
    'cholesterol_hdl': ['SEQN', 'LBDHDD'],
    'glycohemoglobin': ['SEQN', 'LBXGH'],
    'blood_count': ['SEQN', 'LBXWBCSI', 'LBXRBCSI', 'LBXHGB', 'LBXPLTSI'],
    'biochemistry': ['SEQN', 'LBXSGL', 'LBXSCR', 'LBXSUA', 'LBXSAL',
                     'LBXSASSI', 'LBXSATSI', 'LBXSGB', 'LBXSTR'],
    'kidney_albumin_creatinine': ['SEQN', 'URXUMA', 'URXUCR', 'URDACT'],
}

merged_df = base_df.copy()

for component, cols in merge_configs.items():
    if component in nhanes_data:
        # Only keep columns that exist in the data
        available_cols = [c for c in cols if c in nhanes_data[component].columns]
        if 'SEQN' in available_cols:
            subset = nhanes_data[component][available_cols].copy()
            merged_df = merged_df.merge(subset, on='SEQN', how='left')
            print(f'  Merged {component}: +{len(available_cols)-1} columns')
        else:
            print(f'  [SKIP] {component}: SEQN not found')
    else:
        print(f'  [SKIP] {component}: not downloaded')

print(f'\nAfter lab/exam merge: {merged_df.shape}')

  Merged body_measures: +4 columns
  Merged blood_pressure: +4 columns
  Merged cholesterol_total: +1 columns
  Merged cholesterol_hdl: +1 columns
  Merged glycohemoglobin: +1 columns
  Merged blood_count: +4 columns
  Merged biochemistry: +8 columns
  Merged kidney_albumin_creatinine: +3 columns

After lab/exam merge: (30442, 34)


In [14]:
# Merge questionnaire data (smoking, alcohol, diabetes, medical conditions, physical activity)

questionnaire_configs = {
    'smoking': ['SEQN', 'SMQ020', 'SMQ040', 'SMD650'],
    'alcohol': ['SEQN', 'ALQ130', 'ALQ142'],
    'diabetes': ['SEQN', 'DIQ010', 'DIQ050', 'DIQ070'],
    'medical_conditions': ['SEQN', 'MCQ160B', 'MCQ160C', 'MCQ160D',
                           'MCQ160E', 'MCQ160F', 'MCQ220'],
    'physical_activity': ['SEQN', 'PAQ605', 'PAQ620', 'PAQ635', 'PAQ650', 'PAQ665'],
    'health_insurance': ['SEQN', 'HIQ011'],
}

for component, cols in questionnaire_configs.items():
    if component in nhanes_data:
        available_cols = [c for c in cols if c in nhanes_data[component].columns]
        if 'SEQN' in available_cols:
            subset = nhanes_data[component][available_cols].copy()
            merged_df = merged_df.merge(subset, on='SEQN', how='left')
            print(f'  Merged {component}: +{len(available_cols)-1} columns')
    else:
        print(f'  [SKIP] {component}: not downloaded')

print(f'\nAfter questionnaire merge: {merged_df.shape}')

  Merged smoking: +3 columns
  Merged alcohol: +1 columns
  Merged diabetes: +3 columns
  Merged medical_conditions: +6 columns
  Merged physical_activity: +5 columns
  Merged health_insurance: +1 columns

After questionnaire merge: (30442, 53)


In [15]:
# Link mortality outcomes
mortality_subset = mortality_df[['SEQN', 'ELIGSTAT', 'MORTSTAT',
                                  'UCOD_LEADING', 'DIABETES', 'HYPERTEN',
                                  'PERMTH_INT', 'PERMTH_EXM']].copy()

# Rename mortality columns to avoid confusion
mortality_subset.rename(columns={
    'DIABETES': 'MORT_DIABETES',
    'HYPERTEN': 'MORT_HYPERTEN',
}, inplace=True)

final_df = merged_df.merge(mortality_subset, on='SEQN', how='inner')

print(f'Final merged dataset: {final_df.shape}')
print(f'\nMortality status in merged data:')
print(final_df['MORTSTAT'].value_counts(dropna=False))

Final merged dataset: (30442, 60)

Mortality status in merged data:
MORTSTAT
0.0    15963
NaN    11864
1.0     2615
Name: count, dtype: int64


## 1.6 Filter to Eligible Adults & Save

Keep only eligible adults (ELIGSTAT=1) who are 18+ years old.
Save the raw merged dataset for use in subsequent notebooks.

In [16]:
# Filter: Keep only eligible adults (18+)
eligible_df = final_df[
    (final_df['ELIGSTAT'] == 1) & 
    (final_df['AGE'] >= 18)
].copy()

print(f'After filtering to eligible adults (18+): {len(eligible_df)} rows')
print(f'Dropped: {len(final_df) - len(eligible_df)} rows')
print(f'\nMortality in eligible adults:')
print(eligible_df['MORTSTAT'].value_counts(dropna=False))
print(f'\nMortality rate: {eligible_df["MORTSTAT"].mean():.4f} ({eligible_df["MORTSTAT"].sum()} deaths)')

After filtering to eligible adults (18+): 18578 rows
Dropped: 11864 rows

Mortality in eligible adults:
MORTSTAT
0.0    15963
1.0     2615
Name: count, dtype: int64

Mortality rate: 0.1408 (2615.0 deaths)


In [17]:
# Save to processed directory
output_path = PROCESSED_DIR / 'nhanes_mortality_merged.csv'
eligible_df.to_csv(output_path, index=False)
print(f'Saved merged dataset to: {output_path}')
print(f'File size: {output_path.stat().st_size / (1024*1024):.2f} MB')

# Also save as parquet for faster loading later
parquet_path = PROCESSED_DIR / 'nhanes_mortality_merged.parquet'
eligible_df.to_parquet(parquet_path, index=False)
print(f'Saved parquet to: {parquet_path}')
print(f'File size: {parquet_path.stat().st_size / (1024*1024):.2f} MB')

Saved merged dataset to: ..\data\processed\nhanes_mortality_merged.csv
File size: 4.51 MB
Saved parquet to: ..\data\processed\nhanes_mortality_merged.parquet
File size: 0.89 MB


## 1.7 Data Summary

In [18]:
# Quick summary of what we have
print('=' * 60)
print('DATA DOWNLOAD SUMMARY')
print('=' * 60)
print(f'\nDataset shape: {eligible_df.shape}')
print(f'Total respondents: {len(eligible_df)}')
print(f'Total features: {len(eligible_df.columns)}')
print(f'\nCycles included:')
print(eligible_df['CYCLE'].value_counts())
print(f'\nAge statistics:')
print(eligible_df['AGE'].describe())
print(f'\nGender distribution:')
print(eligible_df['GENDER'].value_counts())
print(f'\nMissing values (top 10):')
missing = eligible_df.isnull().sum().sort_values(ascending=False)
print(missing[missing > 0].head(10))
print(f'\nColumn list:')
print(eligible_df.columns.tolist())

DATA DOWNLOAD SUMMARY

Dataset shape: (18578, 60)
Total respondents: 18578
Total features: 60

Cycles included:
CYCLE
2009-2010    6510
2007-2008    6219
2011-2012    5849
Name: count, dtype: int64

Age statistics:
count    18578.000000
mean        48.277263
std         18.734853
min         18.000000
25%         32.000000
50%         48.000000
75%         63.000000
max         80.000000
Name: AGE, dtype: float64

Gender distribution:
GENDER
2.0    9455
1.0    9123
Name: count, dtype: int64

Missing values (top 10):
DIQ070           16415
MORT_DIABETES    15963
UCOD_LEADING     15963
MORT_HYPERTEN    15963
SMD650           14596
SMQ040           10554
ALQ130            8270
URDACT            6888
BPXDI1            2080
BPXSY1            2080
dtype: int64

Column list:
['SEQN', 'CYCLE', 'AGE', 'GENDER', 'RACE_ETHNICITY', 'EDUCATION', 'MARITAL_STATUS', 'POVERTY_INCOME_RATIO', 'BMXBMI', 'BMXWAIST', 'BMXHT', 'BMXWT', 'BPXSY1', 'BPXDI1', 'BPXSY2', 'BPXDI2', 'LBXTC', 'LBDHDD', 'LBXGH', 'LBXW

## Next Steps

The merged dataset is saved at `data/processed/nhanes_mortality_merged.csv` (and `.parquet`).

Proceed to **Notebook 02: Exploratory Data Analysis** for:
- Detailed missing data analysis
- Feature distributions
- Class imbalance investigation
- Correlation analysis